<a href="https://colab.research.google.com/github/IvanBaroni/projects-in-data/blob/main/Apple_Case_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Apple Case Study**
### *• by Ivan Baroni, September 2025*

##**Digital Analytics**

***Objective***: As a Digital Analyst, your objective is to analyze the provided traffic and Ads data to identify opportunities for growth. You must develop a data-driven strategy to leverage sales for the retail partners. Create a presentation to be delivered to the sales and digital teams, justifying your proposed plan with clear insights derived from your analysis.


***Content:*** Partner database with traffic, units sold, impressions, clicks and Ad sold units of Apple
products (dummy data on excel file).
Information to consider:
1. Partner A is a Pure Player Partner;
2. Partner B is an Omnichanell partner;
3. Cyber event during week 35 and 36.

• [Click here](https://docs.google.com/presentation/d/1Fr0rLVEKJqPJGjA_0YmqvpfKs0wdhLV_d-h5xrTugVA/edit?usp=sharing) for the Presentation in Google Slides

# 0 - Importing the Dataset

In [ ]:
import pandas as pd

# Google Docs spreadsheet URL
google_sheet_url = 'https://docs.google.com/spreadsheets/d/1i7tNxgl1YDsvekAUUPLcMKGDfAUQ_EjfLb5d5CtwN2w/edit?usp=sharing'

# Construct the export URL for Excel format
export_url = google_sheet_url.replace('/edit?usp=sharing', '/export?format=xlsx')

# Read the first tab
traffic_df = pd.read_excel(export_url, sheet_name=0)

# Read the second tab
ads_df = pd.read_excel(export_url, sheet_name=1)

print("Traffic dataset")
display(traffic_df.head())

print("\nAds Dataset:")
display(ads_df.head())

Traffic dataset


,Partner,Quarter,Week,Total Site Traffic,Total Sold Units
0,Partner A,Q1,1,3339,84
1,Partner A,Q1,2,3500,88
2,Partner A,Q1,3,3115,78
3,Partner A,Q1,4,3661,92
4,Partner A,Q1,5,3850,97



Ads Dataset:


,Partner,Quarter,Week,Media,Impressions,Clicks,Ad Sold Units
0,Partner A,Q1,1,TikTok,99306,993,17
1,Partner A,Q1,1,Meta,31033,559,17
2,Partner A,Q1,1,Google,15516,621,30
3,Partner A,Q1,2,TikTok,96472,965,17
4,Partner A,Q1,2,Meta,30147,543,16


In [ ]:
def summarize(df, dataset_name="Dataset"):
    bold_blue = '\033[1m\033[94m'
    custom_green = '\033[38;2;50;168;82m'
    reset = '\033[0m'

    print(f"{bold_blue}--- Summary for {dataset_name} ---{reset}")
    df_summary = pd.DataFrame({
        'Null Count': df.isnull().sum(),
        'Unique Count': df.nunique(),
        'Type': df.dtypes
    })
    display(df_summary)
    print(f"{custom_green}The dataframe has {df.shape[0]:,} rows and {df.shape[1]:,} columns.{reset}")


summarize(traffic_df)

--- Summary for Dataset ---


,Null Count,Unique Count,Type
Partner,0,2,object
Quarter,0,4,object
Week,0,52,int64
Total Site Traffic,0,97,int64
Total Sold Units,0,92,int64


The dataframe has 104 rows and 5 columns.


In [ ]:
summarize(ads_df)

--- Summary for Dataset ---


,Null Count,Unique Count,Type
Partner,0,2,object
Quarter,0,4,object
Week,0,52,int64
Media,0,3,object
Impressions,0,230,int64
Clicks,0,218,int64
Ad Sold Units,0,57,int64


The dataframe has 312 rows and 7 columns.


# 1 - Feature Engeneering

### 1.1 Merge the Datasets

In [ ]:
# Pivot the ads_df to have media types as columns
ads_pivot = ads_df.pivot_table(
    index=['Partner', 'Week'],
    columns='Media',
    values=['Impressions', 'Clicks', 'Ad Sold Units']
).reset_index()

# Flatten the multi-level columns
ads_pivot.columns = [f'{col[0]}_{col[1]}' if col[1] else col[0] for col in ads_pivot.columns]

# Merge with traffic_df
merged_df = pd.merge(traffic_df, ads_pivot, on=['Partner', 'Week'], how='left')

merged_df = merged_df.rename(columns={
    "Ad Sold Units_Google": "Google_Ad Sold Units",
    "Ad Sold Units_Meta": "Meta_Ad Sold Units",
    "Ad Sold Units_TikTok": "TikTok_Ad Sold Units",
    "Clicks_Google": "Google_Clicks",
    "Clicks_Meta": "Meta_Clicks",
    "Clicks_TikTok": "TikTok_Clicks",
    "Impressions_Google": "Google_Impressions",
    "Impressions_Meta": "Meta_Impressions",
    "Impressions_TikTok": "TikTok_Impressions",
})

print("Merged DataFrame with pivoted ads data:")
display(merged_df.head())
display(merged_df.shape)

Merged DataFrame with pivoted ads data:


,Partner,Quarter,Week,Total Site Traffic,Total Sold Units,Google_Ad Sold Units,Meta_Ad Sold Units,TikTok_Ad Sold Units,Google_Clicks,Meta_Clicks,TikTok_Clicks,Google_Impressions,Meta_Impressions,TikTok_Impressions
0,Partner A,Q1,1,3339,84,30.0,17.0,17.0,621.0,559.0,993.0,15516.0,31033.0,99306.0
1,Partner A,Q1,2,3500,88,29.0,16.0,17.0,603.0,543.0,965.0,15073.0,30147.0,96472.0
2,Partner A,Q1,3,3115,78,30.0,17.0,17.0,628.0,565.0,1004.0,15692.0,31384.0,100430.0
3,Partner A,Q1,4,3661,92,29.0,16.0,17.0,610.0,549.0,977.0,15245.0,30491.0,97662.0
4,Partner A,Q1,5,3850,97,27.0,15.0,16.0,571.0,514.0,914.0,14277.0,28555.0,91375.0


(104, 14)

### 1.2 Create New Columns

In [ ]:
# Calculate total Impressions, Clicks, and Ad Sold Units (Traffic)
merged_df['Impressions'] = merged_df['Google_Impressions'] + merged_df['TikTok_Impressions'] + merged_df['Meta_Impressions']
merged_df['Clicks'] = merged_df['Google_Clicks'] + merged_df['TikTok_Clicks'] + merged_df['Meta_Clicks']
merged_df['Ad Sold Units'] = merged_df['Google_Ad Sold Units'] + merged_df['TikTok_Ad Sold Units'] + merged_df['Meta_Ad Sold Units']

# Calculate Conversion Rate, CTR, and Ad CVR (Ads)
merged_df['Conversion Rate'] = merged_df['Total Sold Units'] / merged_df['Total Site Traffic']
merged_df['CTR'] = merged_df['Clicks'] / merged_df['Impressions']
merged_df['Ad CVR'] = merged_df['Ad Sold Units'] / merged_df['Impressions']

# Calculate % Traffic from Ads, % units sold from Ads, % Sales from Google, % Sales from TikTok, % Sales from Meta (Merged)
merged_df['% Traffic from Ads'] = merged_df['Clicks'] / merged_df['Total Site Traffic']
merged_df['% units sold from Ads'] = merged_df['Ad Sold Units'] / merged_df['Total Sold Units']
merged_df['% units sold from Google'] = merged_df['Google_Ad Sold Units'] / merged_df['Total Sold Units']
merged_df['% units sold from TikTok'] = merged_df['TikTok_Ad Sold Units'] / merged_df['Total Sold Units']
merged_df['% units sold from Meta'] = merged_df['Meta_Ad Sold Units'] / merged_df['Total Sold Units']

# Set pandas display option to show more rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("DataFrame with new calculated columns:")
display(merged_df.head())

DataFrame with new calculated columns:


,Partner,Quarter,Week,Total Site Traffic,Total Sold Units,Google_Ad Sold Units,Meta_Ad Sold Units,TikTok_Ad Sold Units,Google_Clicks,Meta_Clicks,TikTok_Clicks,Google_Impressions,Meta_Impressions,TikTok_Impressions,Impressions,Clicks,Ad Sold Units,Conversion Rate,CTR,Ad CVR,% Traffic from Ads,% units sold from Ads,% units sold from Google,% units sold from TikTok,% units sold from Meta
0,Partner A,Q1,1,3339,84,30.0,17.0,17.0,621.0,559.0,993.0,15516.0,31033.0,99306.0,145855.0,2173.0,64.0,0.025157,0.014898,0.000439,0.650794,0.761905,0.357143,0.202381,0.202381
1,Partner A,Q1,2,3500,88,29.0,16.0,17.0,603.0,543.0,965.0,15073.0,30147.0,96472.0,141692.0,2111.0,62.0,0.025143,0.014899,0.000438,0.603143,0.704545,0.329545,0.193182,0.181818
2,Partner A,Q1,3,3115,78,30.0,17.0,17.0,628.0,565.0,1004.0,15692.0,31384.0,100430.0,147506.0,2197.0,64.0,0.025040,0.014894,0.000434,0.705297,0.820513,0.384615,0.217949,0.217949
3,Partner A,Q1,4,3661,92,29.0,16.0,17.0,610.0,549.0,977.0,15245.0,30491.0,97662.0,143398.0,2136.0,62.0,0.025130,0.014896,0.000432,0.583447,0.673913,0.315217,0.184783,0.173913
4,Partner A,Q1,5,3850,97,27.0,15.0,16.0,571.0,514.0,914.0,14277.0,28555.0,91375.0,134207.0,1999.0,58.0,0.025195,0.014895,0.000432,0.519221,0.597938,0.278351,0.164948,0.154639


In [ ]:
summarize(merged_df)

--- Summary for Dataset ---


,Null Count,Unique Count,Type
Partner,0,2,object
Quarter,0,4,object
Week,0,52,int64
Total Site Traffic,0,97,int64
Total Sold Units,0,92,int64
Google_Ad Sold Units,0,45,float64
Meta_Ad Sold Units,0,31,float64
TikTok_Ad Sold Units,0,24,float64
Google_Clicks,0,91,float64
Meta_Clicks,0,94,float64


The dataframe has 104 rows and 25 columns.


# 2 - Analyzing Traffic Dataset

In [ ]:
# @title • Site Traffic Evolution

import plotly.express as px

# Calculate the total sold units per week across both partners
total_site_traffic_by_week = merged_df.groupby('Week')['Total Site Traffic'].sum().reset_index()

# Create the interactive trendline chart using Plotly
fig = px.line(total_site_traffic_by_week,
              x='Week',
              y='Total Site Traffic',
              color_discrete_sequence=['#0aab45'],  # Set the line color to green
              title='Total Site Traffic Over Time (Both Partners)', # Updated title to clarify
              hover_data=['Week', 'Total Site Traffic']) # Updated hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1, range=[1, 52]),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Total Site Traffic Over Time (Both Partners)</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size and x-axis range

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=total_site_traffic_by_week['Total Site Traffic'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=total_site_traffic_by_week['Total Site Traffic'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Sold Units Evolution

import plotly.express as px

# Calculate the total sold units per week across both partners
total_sold_units_by_week = merged_df.groupby('Week')['Total Sold Units'].sum().reset_index()

# Create the interactive trendline chart using Plotly
fig = px.line(total_sold_units_by_week,
              x='Week',
              y='Total Sold Units',
              color_discrete_sequence=['#0aab45'],  # Set the line color to green
              title='Total Sold Units Over Time (Both Partners)', # Updated title to clarify
              hover_data=['Week', 'Total Sold Units']) # Updated hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1, range=[1, 52]),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Total Sold Units Over Time (Both Partners)</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size and x-axis range

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=total_sold_units_by_week['Total Sold Units'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=total_sold_units_by_week['Total Sold Units'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Site Traffic by Partners Evolution

import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='Total Site Traffic', # Changed y-axis to 'Total Site Traffic'
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='Total Site Traffic by Partner Over Time', # Changed title
              hover_data=['Partner', 'Week', 'Total Site Traffic']) # Changed hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Total Site Traffic by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['Total Site Traffic'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['Total Site Traffic'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Sold Units by Partners Evolution

import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='Total Sold Units',
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='Total Sold Units by Partner Over Time',
              hover_data=['Partner', 'Week', 'Total Sold Units'])

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Total Sold Units by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['Total Sold Units'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['Total Sold Units'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Conversion Rate Evolution
import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='Conversion Rate', # Changed y-axis to 'Conversion Rate'
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='Conversion Rate by Partner Over Time', # Changed title
              hover_data=['Partner', 'Week', 'Conversion Rate']) # Changed hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 5, dtick = 5),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=800,
                  title=dict(
                      text='<b>Conversion Rate by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['Conversion Rate'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['Conversion Rate'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

# 3 - Analyzing Ads Dataset

In [ ]:
# @title • Impressions Evolution

import plotly.express as px

# Calculate the total sold units per week across both partners
total_impressions_by_week = merged_df.groupby('Week')['Impressions'].sum().reset_index()

# Create the interactive trendline chart using Plotly
fig = px.line(total_impressions_by_week,
              x='Week',
              y='Impressions',
              color_discrete_sequence=['#0aab45'],  # Set the line color to green
              title='Total Impressions Over Time (Both Partners)', # Updated title to clarify
              hover_data=['Week', 'Impressions']) # Updated hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1, range=[1, 52]),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Total Impressions Over Time (Both Partners)</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size and x-axis range

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=total_impressions_by_week['Impressions'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=total_impressions_by_week['Impressions'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Clicks Evolution

import plotly.express as px

# Calculate the total clicks per week across both partners
total_clicks_by_week = merged_df.groupby('Week')['Clicks'].sum().reset_index()

# Create the interactive trendline chart using Plotly
fig = px.line(total_clicks_by_week,
              x='Week',
              y='Clicks',
              color_discrete_sequence=['#0aab45'],  # Set the line color to green
              title='Total Clicks Over Time (Both Partners)', # Updated title to clarify
              hover_data=['Week', 'Clicks']) # Updated hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1, range=[1, 52]),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Total Clicks Over Time (Both Partners)</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size and x-axis range

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=total_clicks_by_week['Clicks'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=total_clicks_by_week['Clicks'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Ad Sold Units Evolution

import plotly.express as px

# Calculate the total Ad Sold Units per week across both partners
total_ad_sold_units_by_week = merged_df.groupby('Week')['Ad Sold Units'].sum().reset_index()

# Create the interactive trendline chart using Plotly
fig = px.line(total_ad_sold_units_by_week,
              x='Week',
              y='Ad Sold Units',
              color_discrete_sequence=['#0aab45'],  # Set the line color to green
              title='Total Ad Sold Units Over Time (Both Partners)', # Updated title to clarify
              hover_data=['Week', 'Ad Sold Units']) # Updated hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1, range=[1, 52]),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Total Ad Sold Units Over Time (Both Partners)</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size and x-axis range

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=total_ad_sold_units_by_week['Ad Sold Units'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=total_ad_sold_units_by_week['Ad Sold Units'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

## 3.1 - Partners deep dive

In [ ]:
# @title • Impression by Partner Evolution
import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='Impressions',
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='Impressions by Partner Over Time',
              hover_data=['Partner', 'Week', 'Impressions'])

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Impressions by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['Impressions'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['Impressions'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Clicks by Partner Evolution
import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='Clicks',
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='Clicks by Partner Over Time',
              hover_data=['Partner', 'Week', 'Clicks'])

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Clicks by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['Clicks'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['Clicks'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)

# Display the chart
fig.show()

In [ ]:
# @title • Ad Sold Units by Partner Evolution
import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='Ad Sold Units',
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='Ad Sold Units by Partner Over Time',
              hover_data=['Partner', 'Week', 'Ad Sold Units'])

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Ad Sold Units by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['Ad Sold Units'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['Ad Sold Units'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • CTR by Partner Evolution
import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='CTR',
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='CTR by Partner Over Time',
              hover_data=['Partner', 'Week', 'CTR'])

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  height=600, width=800, # Changed width to 800
                  title=dict(
                      text='<b>CTR by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  ),
                  plot_bgcolor='#f2f5fb',) # Ensure all integer weeks are shown and set chart size

# Update x-axis to show ticks every 5 weeks, starting from week 5
fig.update_xaxes(tickmode = 'linear', tick0 = 5, dtick = 5, range=[5, 55])


# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['CTR'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['CTR'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Ad CVR by Partner Evolution
import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='Ad CVR',
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='Ad CVR by Partner Over Time',
              hover_data=['Partner', 'Week', 'Ad CVR'])

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 5, dtick = 5),
                  height=600, width=800, # Changed width to 800
                  title=dict(
                      text='<b>Ad CVR by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  ),
                  plot_bgcolor='#f2f5fb',) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['Ad CVR'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['Ad CVR'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

## 3.2 - Media deep dive

In [ ]:
# @title • Impressions by Media Evolution
pivot_table = ads_df.pivot_table(index='Week', columns='Media', values='Impressions', aggfunc='sum')

import plotly.express as px

# Reset index to use 'Week' as a column for plotting
pivot_table_reset = pivot_table.reset_index()

# Melt the DataFrame to long format for Plotly
pivot_table_melted = pivot_table_reset.melt('Week', var_name='Media', value_name='Impressions')

# Create the interactive trendline chart
fig = px.line(pivot_table_melted, x='Week', y='Impressions', color='Media',
              title='Impressions Trend from Pivot Table',
              color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})

fig.update_layout(
    xaxis_title='Week',
    yaxis_title='Impressions',
    hovermode='x unified',
    xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
    plot_bgcolor='#f2f5fb',
    height=600, width=1500
)

fig.update_traces(line=dict(width=5)) # Make lines thicker


# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=380000, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=ads_df['Impressions'].max() * 1.3, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)

# Display the chart
fig.show()

In [ ]:
# @title • Clicks by Media Evolution
pivot_table = ads_df.pivot_table(index='Week', columns='Media', values='Clicks', aggfunc='sum')

import plotly.express as px

# Reset index to use 'Week' as a column for plotting
pivot_table_reset = pivot_table.reset_index()

# Melt the DataFrame to long format for Plotly
pivot_table_melted = pivot_table_reset.melt('Week', var_name='Media', value_name='Clicks')

# Create the interactive trendline chart
fig = px.line(pivot_table_melted, x='Week', y='Clicks', color='Media',
              title='Clicks Trend from Pivot Table',
              color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})

fig.update_layout(
    xaxis_title='Week',
    yaxis_title='Clicks',
    hovermode='x unified',
    xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
    plot_bgcolor='#f2f5fb',
    height=600, width=1500
)

fig.update_traces(line=dict(width=5)) # Make lines thicker


# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=ads_df['Clicks'].max() * 1.3, # Extend slightly above the max y-value, adjusted for Clicks
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=ads_df['Clicks'].max() * 1.2, # Y-coordinate for the text (slightly above the max y-value), adjusted for Clicks
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)

# Display the chart
fig.show()

In [ ]:
# @title • Ad Sold Units by Media
pivot_table = ads_df.pivot_table(index='Week', columns='Media', values='Ad Sold Units', aggfunc='sum')

import plotly.express as px

# Reset index to use 'Week' as a column for plotting
pivot_table_reset = pivot_table.reset_index()

# Melt the DataFrame to long format for Plotly
pivot_table_melted = pivot_table_reset.melt('Week', var_name='Media', value_name='Ad Sold Units')

# Create the interactive trendline chart
fig = px.line(pivot_table_melted, x='Week', y='Ad Sold Units', color='Media',
              title='Ad Sold Units Trend from Pivot Table',
              color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})

fig.update_layout(
    xaxis_title='Week',
    yaxis_title='Ad Sold Units',
    hovermode='x unified',
    xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
    plot_bgcolor='#f2f5fb',
    height=600, width=1500
)

fig.update_traces(line=dict(width=5)) # Make lines thicker

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=ads_df['Ad Sold Units'].max() * 1.3, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=ads_df['Ad Sold Units'].max() * 1.2, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)

# Display the chart
fig.show()

In [ ]:
# @title • CTR by Media Evolution
# Calculate CTR for each media type per week
ads_df['CTR'] = ads_df['Clicks'] / ads_df['Impressions']

# Pivot the data to get CTR for each media type per week
pivot_table = ads_df.pivot_table(index='Week', columns='Media', values='CTR', aggfunc='mean') # Calculate mean CTR for each media type per week

import plotly.express as px

# Reset index to use 'Week' as a column for plotting
pivot_table_reset = pivot_table.reset_index()

# Melt the DataFrame to long format for Plotly
pivot_table_melted = pivot_table_reset.melt('Week', var_name='Media', value_name='CTR')

# Create the interactive trendline chart
fig = px.line(pivot_table_melted, x='Week', y='CTR', color='Media',
              title='CTR Trend by Media Type Over Time',
              color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})

fig.update_layout(
    xaxis_title='Week',
    yaxis_title='CTR',
    hovermode='x unified',
    xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
    plot_bgcolor='#f2f5fb',
    height=600, width=800
)

fig.update_traces(line=dict(width=5)) # Make lines thicker

# Update x-axis to show ticks every 5 weeks, starting from week 5
fig.update_xaxes(tickmode = 'linear', tick0 = 5, dtick = 5, range=[5, 55])


# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=pivot_table_melted['CTR'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=pivot_table_melted['CTR'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Ad CVR byMedia Evolution
# Calculate Ad CVR for each media type per week
ads_df['Ad CVR'] = ads_df['Ad Sold Units'] / ads_df['Impressions']

# Pivot the data to get Ad CVR for each media type per week
pivot_table = ads_df.pivot_table(index='Week', columns='Media', values='Ad CVR', aggfunc='mean') # Calculate mean Ad CVR for each media type per week

import plotly.express as px

# Reset index to use 'Week' as a column for plotting
pivot_table_reset = pivot_table.reset_index()

# Melt the DataFrame to long format for Plotly
pivot_table_melted = pivot_table_reset.melt('Week', var_name='Media', value_name='Ad CVR')

# Create the interactive trendline chart
fig = px.line(pivot_table_melted, x='Week', y='Ad CVR', color='Media',
              title='Ad CVR Trend by Media Type Over Time',
              color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})

fig.update_layout(
    xaxis_title='Week',
    yaxis_title='Ad CVR',
    hovermode='x unified',
    xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
    plot_bgcolor='#f2f5fb',
    height=600, width=800
)

fig.update_traces(line=dict(width=5)) # Make lines thicker

# Update x-axis to show ticks every 5 weeks, starting from week 5
fig.update_xaxes(tickmode = 'linear', tick0 = 5, dtick = 5, range=[5, 55])


# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=pivot_table_melted['Ad CVR'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=pivot_table_melted['Ad CVR'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

# 4 - From Merged dataset

In [ ]:
# @title • Traffic from Ads by Partner Evolution

import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='% Traffic from Ads', # Changed y-axis to '% Traffic from Ads'
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='% Traffic from Ads by Partner Over Time', # Changed title
              hover_data=['Partner', 'Week', '% Traffic from Ads']) # Changed hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>% Traffic from Ads by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['% Traffic from Ads'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['% Traffic from Ads'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Units sold from Ads by Partner Evolution

import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='% units sold from Ads', # Changed y-axis to '% units sold from Ads'
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='% units sold from Ads by Partner Over Time', # Changed title
              hover_data=['Partner', 'Week', '% units sold from Ads']) # Changed hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>% units sold from Ads by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['% units sold from Ads'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['% units sold from Ads'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Units Sold from Ads Evolution

import plotly.express as px

# Calculate the total units sold from ads and total sold units per week across both partners
weekly_data = merged_df.groupby('Week')[['Ad Sold Units', 'Total Sold Units']].sum().reset_index()

# Calculate the overall % units sold from Ads per week
weekly_data['% units sold from Ads'] = weekly_data['Ad Sold Units'] / weekly_data['Total Sold Units']

# Create the interactive trendline chart using Plotly
fig = px.line(weekly_data,
              x='Week',
              y='% units sold from Ads',
              color_discrete_sequence=['#0aab45'],  # Set the line color to green
              title='Overall % units sold from Ads Over Time', # Updated title to clarify
              hover_data=['Week', '% units sold from Ads']) # Updated hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=5)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1, range=[1, 52]),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>Overall % units sold from Ads Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size and x-axis range

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=weekly_data['% units sold from Ads'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=weekly_data['% units sold from Ads'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Units Sold from Google Evolution

import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='% units sold from Google', # Changed y-axis to '% units sold from Google'
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='% units sold from Google by Partner Over Time', # Changed title
              hover_data=['Partner', 'Week', '% units sold from Google']) # Changed hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>% units sold from Google by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['% units sold from Google'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['% units sold from Google'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Units Sold from Meta Evolution

import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='% units sold from Meta', # Changed y-axis to '% units sold from Meta'
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='% units sold from Meta by Partner Over Time', # Changed title
              hover_data=['Partner', 'Week', '% units sold from Meta']) # Changed hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>% units sold from Meta by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['% units sold from Meta'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['% units sold from Meta'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

In [ ]:
# @title • Units Sold from TikTok

import plotly.express as px

# Create the interactive trendline chart using Plotly
fig = px.line(merged_df,
              x='Week',
              y='% units sold from TikTok', # Changed y-axis to '% units sold from TikTok'
              color='Partner',
              color_discrete_map={'Partner A': 'red', 'Partner B': 'blue'},
              title='% units sold from TikTok by Partner Over Time', # Changed title
              hover_data=['Partner', 'Week', '% units sold from TikTok']) # Changed hover data

# Update layout for thicker lines and to ensure all weeks are shown on x-axis
fig.update_traces(line=dict(width=4)) # Make lines thicker
fig.update_layout(xaxis = dict(tickmode = 'linear', tick0 = 1, dtick = 1),
                  plot_bgcolor='#f2f5fb',
                  height=600, width=1500,
                  title=dict(
                      text='<b>% units sold from TikTok by Partner Over Time</b>', # Make title bold
                      xanchor='center',
                      x=0.5,
                      font=dict(size=20) # Increase font size
                  )) # Ensure all integer weeks are shown and set chart size

# Add a rectangle shape to highlight the "CyberWeeks" period
fig.add_shape(
    type="rect",
    x0=35,  # Start of the highlighted period
    y0=0,   # Start from the bottom of the y-axis
    x1=37,  # End of the highlighted period
    y1=merged_df['% units sold from TikTok'].max() * 1.1, # Extend slightly above the max y-value
    fillcolor="lightgreen", # Change fill color to light green
    opacity=0.3, # Make it slightly transparent
    layer="below", # Place the shape below the lines
    line_width=0,
)

# Add annotation for "CyberWeeks"
fig.add_annotation(
    x=36,  # X-coordinate for the text (middle of the highlighted area)
    y=merged_df['% units sold from TikTok'].max() * 1.05, # Y-coordinate for the text (slightly above the max y-value)
    text="CyberWeeks",
    showarrow=False,
    font=dict(
        size=16, # Increase font size
        color="green",
        family="Arial",
        weight="bold"
    ),
    xanchor="center",
    yanchor="bottom"
)


# Display the chart
fig.show()

# 5 - CyberWeeks Analysis

In [ ]:
# Create a pivot table with Week as index and specified columns as values
pivot_table = merged_df.pivot_table(
    index='Week',
    values=['Total Site Traffic', 'Total Sold Units', 'Impressions', 'Clicks', 'Ad Sold Units'],
    aggfunc='sum'  # Assuming you want the sum of these values per week
)

# Display the pivot table
print("Pivot Table based on merged_df:")
display(pivot_table.head())

Pivot Table based on merged_df:


,Ad Sold Units,Clicks,Impressions,Total Site Traffic,Total Sold Units
Week,,,,,
1,82.0,3043.0,278855.0,4770,105
2,80.0,2955.0,270492.0,5000,110
3,82.0,3076.0,281606.0,4450,98
4,80.0,2989.0,273298.0,5230,115
5,74.0,2799.0,256607.0,5500,121


In [ ]:
# Define the metrics to include in the table
metrics = ['Total Site Traffic', 'Total Sold Units', 'Impressions', 'Clicks', 'Ad Sold Units']

# Calculate the average for the 4 weeks before CyberWeeks (Week 31-34)
pre_Cyber_Weeks_avg = pivot_table[(pivot_table.index >= 31) & (pivot_table.index <= 34)][metrics].mean()

# Get the values for CyberWeeks (Week 35 and 36)
Cyber_Weeks_values = pivot_table[(pivot_table.index >= 35) & (pivot_table.index <= 36)][metrics].mean()

# Calculate the average for the 4 weeks after CyberWeeks (Week 37-40)
post_Cyber_Weeks_avg = pivot_table[(pivot_table.index >= 37) & (pivot_table.index <= 40)][metrics].mean()

# Create a new DataFrame for the comparison table
comparison_table = pd.DataFrame({
    'Pre (Avg Weeks 31-34)': pre_Cyber_Weeks_avg,
    'CyberWeeks (Avg Weeks 35-36)': Cyber_Weeks_values,
    'Post (Avg Weeks 37-40)': post_Cyber_Weeks_avg
})

# Add a new column for percentage increase entering CyberWeeks
comparison_table['Entering CyberWeeks (%)'] = ((comparison_table['CyberWeeks (Avg Weeks 35-36)'] - comparison_table['Pre (Avg Weeks 31-34)']) / comparison_table['Pre (Avg Weeks 31-34)']) * 100

# Add a new column for percentage change leaving CyberWeeks
comparison_table['Leaving CyberWeeks (%)'] = ((comparison_table['Post (Avg Weeks 37-40)'] - comparison_table['CyberWeeks (Avg Weeks 35-36)']) / comparison_table['CyberWeeks (Avg Weeks 35-36)']) * 100

# Function to color text based on value
def color_negative_positive(value):
    if value < 0:
        return 'color: red'
    elif value > 0:
        return 'color: green'
    else:
        return ''

print("Comparison of Metrics: Pre-CyberWeeks, CyberWeeks, and Post-CyberWeeks")
display(comparison_table.style.format(thousands=',', precision=1).map(color_negative_positive, subset=['Entering CyberWeeks (%)', 'Leaving CyberWeeks (%)']).set_properties(subset=['Entering CyberWeeks (%)', 'Leaving CyberWeeks (%)'], **{'text-align': 'center', 'font-weight': 'bold'}))

Comparison of Metrics: Pre-CyberWeeks, CyberWeeks, and Post-CyberWeeks


,Pre (Avg Weeks 31-34),CyberWeeks (Avg Weeks 35-36),Post (Avg Weeks 37-40),Entering CyberWeeks (%),Leaving CyberWeeks (%)
Total Site Traffic,"12,250.0","12,400.0","12,425.0",1.2,0.2
Total Sold Units,268.8,272.0,272.5,1.2,0.2
Impressions,"546,430.2","703,596.5","563,131.0",28.8,-20.0
Clicks,"6,138.2","7,932.0","6,356.5",29.2,-19.9
Ad Sold Units,167.0,216.5,173.5,29.6,-19.9


# 6 - Correlation Matrix

## 6.1 Pivot Tables

In [ ]:
# Create a pivot table with Week as index and specified columns as values for Partner A
pivot_correl = merged_df.pivot_table(
    index='Week',
    values=[
        'Total Site Traffic',
        'Total Sold Units',
        'Google_Impressions',
        'Google_Clicks',
        'Google_Ad Sold Units',
        'Meta_Impressions',
        'Meta_Clicks',
        'Meta_Ad Sold Units',
        'TikTok_Impressions',
        'TikTok_Clicks',
        'TikTok_Ad Sold Units',
        'Impressions',
        'Clicks',
        'Ad Sold Units'
    ],
    aggfunc='sum'  # Assuming you want the sum of these values per week
)

# Display the pivot table
print("Pivot Table based on merged_df:")
display(pivot_correl.head())

Pivot Table based on merged_df:


,Ad Sold Units,Clicks,Google_Ad Sold Units,Google_Clicks,Google_Impressions,Impressions,Meta_Ad Sold Units,Meta_Clicks,Meta_Impressions,TikTok_Ad Sold Units,TikTok_Clicks,TikTok_Impressions,Total Site Traffic,Total Sold Units
Week,,,,,,,,,,,,,,
1,82.0,3043.0,40.0,1021.0,23516.0,278855.0,25.0,1019.0,146033.0,17.0,1003.0,109306.0,4770,105
2,80.0,2955.0,39.0,993.0,22873.0,270492.0,24.0,987.0,141147.0,17.0,975.0,106472.0,5000,110
3,82.0,3076.0,40.0,1033.0,23792.0,281606.0,25.0,1029.0,147384.0,17.0,1014.0,110430.0,4450,98
4,80.0,2989.0,39.0,1005.0,23145.0,273298.0,24.0,997.0,142491.0,17.0,987.0,107662.0,5230,115
5,74.0,2799.0,36.0,941.0,21677.0,256607.0,22.0,934.0,133555.0,16.0,924.0,101375.0,5500,121


In [ ]:
# Filter the data for Partner A
partner_a_df = merged_df[merged_df['Partner'] == 'Partner A'].copy()

# Create a pivot table with Week as index and specified columns as values for Partner A
pivot_correl_a = partner_a_df.pivot_table(
    index='Week',
    values=[
        'Total Site Traffic',
        'Total Sold Units',
        'Google_Impressions',
        'Google_Clicks',
        'Google_Ad Sold Units',
        'Meta_Impressions',
        'Meta_Clicks',
        'Meta_Ad Sold Units',
        'TikTok_Impressions',
        'TikTok_Clicks',
        'TikTok_Ad Sold Units',
        'Impressions',
        'Clicks',
        'Ad Sold Units'
    ],
    aggfunc='sum'  # Assuming you want the sum of these values per week
)

# Display the pivot table
print("Pivot Table based on merged_df for Partner A:")
display(pivot_correl_a.head())

Pivot Table based on merged_df for Partner A:


,Ad Sold Units,Clicks,Google_Ad Sold Units,Google_Clicks,Google_Impressions,Impressions,Meta_Ad Sold Units,Meta_Clicks,Meta_Impressions,TikTok_Ad Sold Units,TikTok_Clicks,TikTok_Impressions,Total Site Traffic,Total Sold Units
Week,,,,,,,,,,,,,,
1,64.0,2173.0,30.0,621.0,15516.0,145855.0,17.0,559.0,31033.0,17.0,993.0,99306.0,3339,84
2,62.0,2111.0,29.0,603.0,15073.0,141692.0,16.0,543.0,30147.0,17.0,965.0,96472.0,3500,88
3,64.0,2197.0,30.0,628.0,15692.0,147506.0,17.0,565.0,31384.0,17.0,1004.0,100430.0,3115,78
4,62.0,2136.0,29.0,610.0,15245.0,143398.0,16.0,549.0,30491.0,17.0,977.0,97662.0,3661,92
5,58.0,1999.0,27.0,571.0,14277.0,134207.0,15.0,514.0,28555.0,16.0,914.0,91375.0,3850,97


In [ ]:
# Filter the data for Partner B
partner_b_df = merged_df[merged_df['Partner'] == 'Partner B'].copy()

# Create a pivot table with Week as index and specified columns as values for Partner B
pivot_correl_b = partner_b_df.pivot_table(
    index='Week',
    values=[
        'Total Site Traffic',
        'Total Sold Units',
        'Google_Impressions',
        'Google_Clicks',
        'Google_Ad Sold Units',
        'Meta_Impressions',
        'Meta_Clicks',
        'Meta_Ad Sold Units',
        'TikTok_Impressions',
        'TikTok_Clicks',
        'TikTok_Ad Sold Units',
        'Impressions',
        'Clicks',
        'Ad Sold Units'
    ],
    aggfunc='sum'  # Assuming you want the sum of these values per week
)

# Display the pivot table
print("Pivot Table based on merged_df for Partner B:")
display(pivot_correl_b.head())

Pivot Table based on merged_df for Partner B:


,Ad Sold Units,Clicks,Google_Ad Sold Units,Google_Clicks,Google_Impressions,Impressions,Meta_Ad Sold Units,Meta_Clicks,Meta_Impressions,TikTok_Ad Sold Units,TikTok_Clicks,TikTok_Impressions,Total Site Traffic,Total Sold Units
Week,,,,,,,,,,,,,,
1,18.0,870.0,10.0,400.0,8000.0,133000.0,8.0,460.0,115000.0,0.0,10.0,10000.0,1431,21
2,18.0,844.0,10.0,390.0,7800.0,128800.0,8.0,444.0,111000.0,0.0,10.0,10000.0,1500,22
3,18.0,879.0,10.0,405.0,8100.0,134100.0,8.0,464.0,116000.0,0.0,10.0,10000.0,1335,20
4,18.0,853.0,10.0,395.0,7900.0,129900.0,8.0,448.0,112000.0,0.0,10.0,10000.0,1569,23
5,16.0,800.0,9.0,370.0,7400.0,122400.0,7.0,420.0,105000.0,0.0,10.0,10000.0,1650,24


## 6.2 Matrix Correlation Tables

In [ ]:
# Select only the desired metrics
selected_metrics = ['Total Site Traffic', 'Total Sold Units', 'Impressions', 'Clicks', 'Ad Sold Units']
filtered_df = pivot_correl[selected_metrics]

# Calculate the correlation matrix for the selected numerical columns
correlation_matrix = filtered_df.corr(numeric_only=True)

# Display the correlation matrix with heatmap styling
print("Correlation Matrix:")
display(correlation_matrix.style.background_gradient(cmap='Greens').format(precision=2).set_properties(**{'text-align': 'center', 'font-weight': 'bold', 'border': '1px solid white', 'width': '100px'}))

Correlation Matrix:


,Total Site Traffic,Total Sold Units,Impressions,Clicks,Ad Sold Units
Total Site Traffic,1.00,1.00,0.81,0.82,0.82
Total Sold Units,1.00,1.00,0.81,0.82,0.82
Impressions,0.81,0.81,1.00,1.00,1.00
Clicks,0.82,0.82,1.00,1.00,1.00
Ad Sold Units,0.82,0.82,1.00,1.00,1.00


In [ ]:
# Select only the desired metrics
selected_metrics = [
    'Total Site Traffic',
    'Total Sold Units',
    'Google_Impressions',
    'Google_Clicks',
    'Google_Ad Sold Units',
    'Meta_Impressions',
    'Meta_Clicks',
    'Meta_Ad Sold Units',
    'TikTok_Impressions',
    'TikTok_Clicks',
    'TikTok_Ad Sold Units'
]
filtered_df = pivot_correl[selected_metrics]

# Calculate the correlation matrix for the selected numerical columns
correlation_matrix = filtered_df.corr(numeric_only=True)

# Display the correlation matrix with heatmap styling, 3 decimal places, centered and bold numbers
print("Correlation Matrix:")
display(correlation_matrix.style.background_gradient(cmap='Greens').format(precision=2).set_properties(**{'text-align': 'center', 'font-weight': 'bold', 'border': '1px solid white', 'width': '120px'}))

Correlation Matrix:


,Total Site Traffic,Total Sold Units,Google_Impressions,Google_Clicks,Google_Ad Sold Units,Meta_Impressions,Meta_Clicks,Meta_Ad Sold Units,TikTok_Impressions,TikTok_Clicks,TikTok_Ad Sold Units
Total Site Traffic,1.00,1.00,0.82,0.82,0.82,0.79,0.81,0.82,0.83,0.83,0.83
Total Sold Units,1.00,1.00,0.82,0.82,0.82,0.79,0.81,0.82,0.83,0.83,0.83
Google_Impressions,0.82,0.82,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Google_Clicks,0.82,0.82,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Google_Ad Sold Units,0.82,0.82,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Meta_Impressions,0.79,0.79,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Meta_Clicks,0.81,0.81,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Meta_Ad Sold Units,0.82,0.82,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
TikTok_Impressions,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
TikTok_Clicks,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


In [ ]:
# Select only the desired metrics
selected_metrics = ['Total Site Traffic', 'Total Sold Units', 'Impressions', 'Clicks', 'Ad Sold Units']
filtered_df = pivot_correl_a[selected_metrics]

# Calculate the correlation matrix for the selected numerical columns
correlation_matrix = filtered_df.corr(numeric_only=True)

# Display the correlation matrix for Partner A
print("Correlation Matrix for Partner A:")
display(correlation_matrix.style.background_gradient(cmap='Reds').format(precision=2).set_properties(**{'text-align': 'center', 'font-weight': 'bold', 'border': '1px solid white', 'width': '100px'}))

Correlation Matrix for Partner A:


,Total Site Traffic,Total Sold Units,Impressions,Clicks,Ad Sold Units
Total Site Traffic,1.00,1.00,0.83,0.83,0.83
Total Sold Units,1.00,1.00,0.83,0.83,0.83
Impressions,0.83,0.83,1.00,1.00,1.00
Clicks,0.83,0.83,1.00,1.00,1.00
Ad Sold Units,0.83,0.83,1.00,1.00,1.00


In [ ]:
# Select only the desired metrics
selected_metrics = [
    'Total Site Traffic',
    'Total Sold Units',
    'Google_Impressions',
    'Google_Clicks',
    'Google_Ad Sold Units',
    'Meta_Impressions',
    'Meta_Clicks',
    'Meta_Ad Sold Units',
    'TikTok_Impressions',
    'TikTok_Clicks',
    'TikTok_Ad Sold Units'
]
filtered_df = pivot_correl_a[selected_metrics]

# Calculate the correlation matrix for the selected numerical columns
correlation_matrix = filtered_df.corr(numeric_only=True)

# Display the correlation matrix with heatmap styling,
print("Correlation Matrix for Partner A:")
display(correlation_matrix.style.background_gradient(cmap='Reds').format(precision=2).set_properties(**{'text-align': 'center', 'font-weight': 'bold', 'border': '1px solid white', 'width': '120px'}))

Correlation Matrix for Partner A:


,Total Site Traffic,Total Sold Units,Google_Impressions,Google_Clicks,Google_Ad Sold Units,Meta_Impressions,Meta_Clicks,Meta_Ad Sold Units,TikTok_Impressions,TikTok_Clicks,TikTok_Ad Sold Units
Total Site Traffic,1.00,1.00,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83
Total Sold Units,1.00,1.00,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83
Google_Impressions,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Google_Clicks,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Google_Ad Sold Units,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Meta_Impressions,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Meta_Clicks,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Meta_Ad Sold Units,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
TikTok_Impressions,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
TikTok_Clicks,0.83,0.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


In [ ]:
# Select only the desired metrics
selected_metrics = ['Total Site Traffic', 'Total Sold Units', 'Impressions', 'Clicks', 'Ad Sold Units']
filtered_df = pivot_correl_b[selected_metrics]

# Calculate the correlation matrix for the selected numerical columns
correlation_matrix = filtered_df.corr(numeric_only=True)

# Display the correlation matrix for Partner A
print("Correlation Matrix for Partner B:")
display(correlation_matrix.style.background_gradient(cmap='Blues').format(precision=2).set_properties(**{'text-align': 'center', 'font-weight': 'bold', 'border': '1px solid white', 'width': '100px'}))

Correlation Matrix for Partner B:


,Total Site Traffic,Total Sold Units,Impressions,Clicks,Ad Sold Units
Total Site Traffic,1.00,1.00,0.78,0.79,0.79
Total Sold Units,1.00,1.00,0.77,0.78,0.79
Impressions,0.78,0.77,1.00,1.00,1.00
Clicks,0.79,0.78,1.00,1.00,1.00
Ad Sold Units,0.79,0.79,1.00,1.00,1.00


In [ ]:
# Select only the desired metrics
selected_metrics = [
    'Total Site Traffic',
    'Total Sold Units',
    'Google_Impressions',
    'Google_Clicks',
    'Google_Ad Sold Units',
    'Meta_Impressions',
    'Meta_Clicks',
    'Meta_Ad Sold Units',
    'TikTok_Impressions',
    'TikTok_Clicks',
    'TikTok_Ad Sold Units'
]
filtered_df = pivot_correl_b[selected_metrics]

# Calculate the correlation matrix for the selected numerical columns
correlation_matrix = filtered_df.corr(numeric_only=True)

# Display the correlation matrix with heatmap styling,
print("Correlation Matrix for Partner B:")
display(correlation_matrix.style.background_gradient(cmap='Blues').format(precision=2).set_properties(**{'text-align': 'center', 'font-weight': 'bold', 'border': '1px solid white', 'width': '120px'}))

Correlation Matrix for Partner B:


/usr/local/lib/python3.12/dist-packages/pandas/io/formats/style.py:3807: RuntimeWarning:

All-NaN slice encountered

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/style.py:3808: RuntimeWarning:

All-NaN slice encountered



,Total Site Traffic,Total Sold Units,Google_Impressions,Google_Clicks,Google_Ad Sold Units,Meta_Impressions,Meta_Clicks,Meta_Ad Sold Units,TikTok_Impressions,TikTok_Clicks,TikTok_Ad Sold Units
Total Site Traffic,1.00,1.00,0.81,0.81,0.80,0.77,0.77,0.77,nan,nan,nan
Total Sold Units,1.00,1.00,0.80,0.80,0.80,0.77,0.77,0.76,nan,nan,nan
Google_Impressions,0.81,0.80,1.00,1.00,1.00,1.00,1.00,0.99,nan,nan,nan
Google_Clicks,0.81,0.80,1.00,1.00,1.00,1.00,1.00,0.99,nan,nan,nan
Google_Ad Sold Units,0.80,0.80,1.00,1.00,1.00,0.99,0.99,0.99,nan,nan,nan
Meta_Impressions,0.77,0.77,1.00,1.00,0.99,1.00,1.00,1.00,nan,nan,nan
Meta_Clicks,0.77,0.77,1.00,1.00,0.99,1.00,1.00,1.00,nan,nan,nan
Meta_Ad Sold Units,0.77,0.76,0.99,0.99,0.99,1.00,1.00,1.00,nan,nan,nan
TikTok_Impressions,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
TikTok_Clicks,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


# 7 - Efficiency Rates

## 7.1 Bar Charts from Traffic Dataset

In [ ]:
# @title • Conversion Rate
# Calculate overall Conversion Rate
overall_conversion_rate = merged_df['Total Sold Units'].sum() / merged_df['Total Site Traffic'].sum()

# Calculate Conversion Rate for Partner A
partner_a_conversion_rate = merged_df[merged_df['Partner'] == 'Partner A']['Total Sold Units'].sum() / merged_df[merged_df['Partner'] == 'Partner A']['Total Site Traffic'].sum()

# Calculate Conversion Rate for Partner B
partner_b_conversion_rate = merged_df[merged_df['Partner'] == 'Partner B']['Total Sold Units'].sum() / merged_df[merged_df['Partner'] == 'Partner B']['Total Site Traffic'].sum()


# Create a DataFrame for plotting
conversion_rate_data = pd.DataFrame({
    'Category': ['Overall', 'Partner A', 'Partner B'],
    'Conversion Rate': [overall_conversion_rate, partner_a_conversion_rate, partner_b_conversion_rate]
})

# Create the bar chart
fig = px.bar(conversion_rate_data,
             x='Category',
             y='Conversion Rate',
             title='Comparison of Conversion Rate: Overall, Partner A, and Partner B',
             text=conversion_rate_data['Conversion Rate'].apply(lambda x: f'<b>{x:.2%}</b>')) # Add text to show values on bars

# Update layout
fig.update_layout(
    xaxis_title=dict(text='Category', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    # yaxis_title='CTR', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Comparison of Conversion Rate</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=700, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition='inside', textfont=dict(size=25))

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', 'red', 'blue']
bar_line_colors = ['grey', 'red', 'blue']
bar_line_widths = [4, 0, 0]
text_colors = ['grey', 'white', 'white']


# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))

# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=2.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

## 7.2 Bar Charts from Ads Dataset

In [ ]:
# @title • CTR
# Calculate overall CTR
overall_ctr = merged_df['Clicks'].sum() / merged_df['Impressions'].sum()

# Calculate CTR for Partner A
partner_a_ctr = merged_df[merged_df['Partner'] == 'Partner A']['Clicks'].sum() / merged_df[merged_df['Partner'] == 'Partner A']['Impressions'].sum()

# Calculate CTR for Partner B
partner_b_ctr = merged_df[merged_df['Partner'] == 'Partner B']['Clicks'].sum() / merged_df[merged_df['Partner'] == 'Partner B']['Impressions'].sum()

# Create a DataFrame for plotting
ctr_data = pd.DataFrame({
    'Category': ['Overall', 'Partner A', 'Partner B'],
    'CTR': [overall_ctr, partner_a_ctr, partner_b_ctr]
})

# Create the bar chart
fig = px.bar(ctr_data,
             x='Category',
             y='CTR',
             title='Comparison of CTR: Overall, Partner A, and Partner B',
             text=ctr_data['CTR'].apply(lambda x: f'<b>{x:.2%}</b>')) # Add text to show values on bars

# Update layout
fig.update_layout(
    xaxis_title=dict(text='Category', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    # yaxis_title='CTR', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Comparison of CTR: Overall, Partner A, and Partner B</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=700, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition='inside', textfont=dict(size=25)) # Increased text font size

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', 'red', 'blue']
bar_line_colors = ['grey', 'red', 'blue']
bar_line_widths = [4, 0, 0]
text_colors = ['grey', 'white', 'white']


# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))

# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=2.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="black",
        width=2,
    )
)


fig.show()

In [ ]:
# @title • Ad CVR
# Calculate overall Ad CVR
overall_ad_cvr = merged_df['Ad Sold Units'].sum() / merged_df['Impressions'].sum()

# Calculate Ad CVR for Partner A
partner_a_ad_cvr = merged_df[merged_df['Partner'] == 'Partner A']['Ad Sold Units'].sum() / merged_df[merged_df['Partner'] == 'Partner A']['Impressions'].sum()

# Calculate Ad CVR for Partner B
partner_b_ad_cvr = merged_df[merged_df['Partner'] == 'Partner B']['Ad Sold Units'].sum() / merged_df[merged_df['Partner'] == 'Partner B']['Impressions'].sum()

# Create a DataFrame for plotting
ad_cvr_data = pd.DataFrame({
    'Category': ['Overall', 'Partner A', 'Partner B'],
    'Ad CVR': [overall_ad_cvr, partner_a_ad_cvr, partner_b_ad_cvr]
})

# Create the bar chart
fig = px.bar(ad_cvr_data,
             x='Category',
             y='Ad CVR',
             title='Comparison of Ad CVR: Overall, Partner A, and Partner B',
             text=ad_cvr_data['Ad CVR'].apply(lambda x: f'<b>{x:.2%}</b>')) # Add text to show values on bars

# Update layout
fig.update_layout(
    xaxis_title=dict(text='Category', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    # yaxis_title='CTR', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Comparison of Ad CVR</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=700, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition='inside', textfont=dict(size=25)) # Increased text font size

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', 'red', 'blue']
bar_line_colors = ['grey', 'red', 'blue']
bar_line_widths = [4, 0, 0]
text_colors = ['grey', 'white', 'white']


# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))

# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=2.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

## 7.3 Bar Charts for Merged Dataset

In [ ]:
# @title • % Traffic from Ads
# Calculate overall % Traffic from Ads
overall_traffic_from_ads = merged_df['Clicks'].sum() / merged_df['Total Site Traffic'].sum()

# Calculate % Traffic from Ads for Partner A
partner_a_traffic_from_ads = merged_df[merged_df['Partner'] == 'Partner A']['Clicks'].sum() / merged_df[merged_df['Partner'] == 'Partner A']['Total Site Traffic'].sum()

# Calculate % Traffic from Ads for Partner B
partner_b_traffic_from_ads = merged_df[merged_df['Partner'] == 'Partner B']['Clicks'].sum() / merged_df[merged_df['Partner'] == 'Partner B']['Total Site Traffic'].sum()

# Create a DataFrame for plotting
traffic_from_ads_data = pd.DataFrame({
    'Category': ['Overall', 'Partner A', 'Partner B'],
    '% Traffic from Ads': [overall_traffic_from_ads, partner_a_traffic_from_ads, partner_b_traffic_from_ads]
})

# Create the bar chart
fig = px.bar(traffic_from_ads_data,
             x='Category',
             y='% Traffic from Ads',
             title='Comparison of % Traffic from Ads: Overall, Partner A, and Partner B',
             text=traffic_from_ads_data['% Traffic from Ads'].apply(lambda x: f'<b>{x:.1%}</b>')) # Add text to show values on bars

# Update layout
fig.update_layout(
    xaxis_title=dict(text='Category', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    # yaxis_title='CTR', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Comparison of % Traffic from Ads</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=700, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition='inside', textfont=dict(size=25)) # Increased text font size

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', 'red', 'blue']
bar_line_colors = ['grey', 'red', 'blue']
bar_line_widths = [4, 0, 0]
text_colors = ['grey', 'white', 'white']


# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))


# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=2.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

In [ ]:
# @title • % Units Sold from Ads
# Calculate overall % units sold from Ads
overall_unit_solds_from_ads = merged_df['Ad Sold Units'].sum() / merged_df['Total Sold Units'].sum()

# Calculate % units sold from Ads for Partner A
partner_a_unit_solds_from_ads = merged_df[merged_df['Partner'] == 'Partner A']['Ad Sold Units'].sum() / merged_df[merged_df['Partner'] == 'Partner A']['Total Sold Units'].sum()

# Calculate % units sold from Ads for Partner B
partner_b_unit_solds_from_ads = merged_df[merged_df['Partner'] == 'Partner B']['Ad Sold Units'].sum() / merged_df[merged_df['Partner'] == 'Partner B']['Total Sold Units'].sum()

# Create a DataFrame for plotting
unit_solds_from_ads_data = pd.DataFrame({
    'Category': ['Overall', 'Partner A', 'Partner B'],
    '% units sold from Ads': [overall_unit_solds_from_ads, partner_a_unit_solds_from_ads, partner_b_unit_solds_from_ads]
})

# Create the bar chart
fig = px.bar(unit_solds_from_ads_data,
             x='Category',
             y='% units sold from Ads',
             title='Comparison of % units sold from Ads: Overall, Partner A, and Partner B',
             text=unit_solds_from_ads_data['% units sold from Ads'].apply(lambda x: f'<b>{x:.1%}</b>')) # Add text to show values on bars

# Update layout
fig.update_layout(
    xaxis_title=dict(text='Category', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    # yaxis_title='CTR', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Comparison of % units sold from Ads</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=700, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition='inside', textfont=dict(size=25))

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', 'red', 'blue']
bar_line_colors = ['grey', 'red', 'blue']
bar_line_widths = [4, 0, 0]
text_colors = ['grey', 'white', 'white']


# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))

# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=2.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

# 8 - Media Efficiency Rates

## 8.1 Media Bar Charts

In [ ]:
# @title • CTR
# Calculate CTR for each media type
media_ctr = ads_df.groupby('Media').apply(lambda x: x['Clicks'].sum() / x['Impressions'].sum(), include_groups=False).reset_index(name='CTR')

# Calculate overall CTR for all media
overall_ctr = ads_df['Clicks'].sum() / ads_df['Impressions'].sum()

# Create a DataFrame for plotting including overall CTR
ctr_data = pd.DataFrame({
    'Media': ['Overall'] + media_ctr['Media'].tolist(),
    'CTR': [overall_ctr] + media_ctr['CTR'].tolist()
})

# Create the bar chart
fig = px.bar(ctr_data,
             x='Media',
             y='CTR',
             title='CTR by Media Type',
             text=ctr_data['CTR'].apply(lambda x: f'<b>{x:.2%}</b>')) # Add text to show values on bars


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Media Type', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>CTR by Media Type</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=900, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition='inside', textfont=dict(size=25)) # Changed text color to white

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', '#A0D568', '#4fc1e8', '#ac92eb']
bar_line_colors = ['grey', '#ffce54', '#4fc1e8', '#ac92eb']
bar_line_widths = [4, 0, 0, 0]
text_colors = ['grey', 'white', 'white', 'white']

# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))


# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=3.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)

fig.show()

In [ ]:
# @title • CTR for Partner A
# Filter ads_df for Partner A
partner_a_ads_df = ads_df[ads_df['Partner'] == 'Partner A'].copy()

# Calculate CTR for each media type for Partner A
partner_a_media_ctr = partner_a_ads_df.groupby('Media').apply(lambda x: x['Clicks'].sum() / x['Impressions'].sum(), include_groups=False).reset_index(name='CTR')

# Calculate overall CTR for all media for Partner A
partner_a_overall_ctr = partner_a_ads_df['Clicks'].sum() / partner_a_ads_df['Impressions'].sum()

# Create a DataFrame for plotting including overall CTR for Partner A
partner_a_ctr_data = pd.DataFrame({
    'Media': ['Overall'] + partner_a_media_ctr['Media'].tolist(),
    'CTR': [partner_a_overall_ctr] + partner_a_media_ctr['CTR'].tolist()
})

# Create the bar chart
fig = px.bar(partner_a_ctr_data,
             x='Media',
             y='CTR',
             title='CTR by Media Type for Partner A',
             text=partner_a_ctr_data['CTR'].apply(lambda x: f'<b>{x:.2%}</b>')) # Add text to show values on bars


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Media Type', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>CTR by Media Type for Partner A</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=900, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition=['inside', 'inside', 'inside', 'outside'], textfont=dict(size=25))

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', '#A0D568', '#4fc1e8', '#ac92eb']
bar_line_colors = ['grey', '#ffce54', '#4fc1e8', '#ac92eb']
bar_line_widths = [4, 0, 0, 0]
text_colors = ['grey', 'white', 'white', '#ac92eb']

# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))


# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=3.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

In [ ]:
# @title • CTR for Partner B
# Filter ads_df for Partner B
partner_b_ads_df = ads_df[ads_df['Partner'] == 'Partner B'].copy()

# Calculate CTR for each media type for Partner B
partner_b_media_ctr = partner_b_ads_df.groupby('Media').apply(lambda x: x['Clicks'].sum() / x['Impressions'].sum(), include_groups=False).reset_index(name='CTR')

# Calculate overall CTR for all media for Partner B
partner_b_overall_ctr = partner_b_ads_df['Clicks'].sum() / partner_b_ads_df['Impressions'].sum()

# Create a DataFrame for plotting including overall CTR for Partner B
partner_b_ctr_data = pd.DataFrame({
    'Media': ['Overall'] + partner_b_media_ctr['Media'].tolist(),
    'CTR': [partner_b_overall_ctr] + partner_b_media_ctr['CTR'].tolist()
})

# Create the bar chart
fig = px.bar(partner_b_ctr_data,
             x='Media',
             y='CTR',
             title='CTR by Media Type for Partner B',
             text=partner_b_ctr_data['CTR'].apply(lambda x: f'<b>{x:.2%}</b>')) # Add text to show values on bars


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Media Type', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>CTR by Media Type for Partner B</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=900, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition=['inside', 'inside', 'outside', 'outside'], textfont=dict(size=25)) # Changed text color to white

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', '#A0D568', '#4fc1e8', '#ac92eb']
bar_line_colors = ['grey', '#ffce54', '#4fc1e8', '#ac92eb']
bar_line_widths = [4, 0, 0, 0]
text_colors = ['grey', 'white', '#4fc1e8', '#ac92eb'] # Changed text color for the last bar

# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))


# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=3.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

In [ ]:
# @title • Ad CVR
# Calculate Ad CVR for each media type
media_ad_cvr = ads_df.groupby('Media').apply(lambda x: x['Ad Sold Units'].sum() / x['Impressions'].sum(), include_groups=False).reset_index(name='Ad CVR')

# Calculate overall Ad CVR for all media
overall_ad_cvr = ads_df['Ad Sold Units'].sum() / ads_df['Impressions'].sum()

# Create a DataFrame for plotting including overall Ad CVR
ad_cvr_data = pd.DataFrame({
    'Media': ['Overall', 'Google', 'Meta', 'TikTok'],
    'Ad CVR': [overall_ad_cvr, media_ad_cvr[media_ad_cvr['Media'] == 'Google']['Ad CVR'].iloc[0], media_ad_cvr[media_ad_cvr['Media'] == 'Meta']['Ad CVR'].iloc[0], media_ad_cvr[media_ad_cvr['Media'] == 'TikTok']['Ad CVR'].iloc[0]]
})

# Create the bar chart
fig = px.bar(ad_cvr_data,
             x='Media',
             y='Ad CVR',
             title='Ad CVR by Media Type',
             text=ad_cvr_data.apply(lambda row: f'<b>{row["Ad CVR"]:.2%}</b>' if row['Media'] in ['Overall', 'Google'] else f'<b>{row["Ad CVR"]:.3%}</b>', axis=1)) # Apply conditional formatting


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Media Type', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Ad CVR by Media Type</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=900, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition=['inside', 'inside', 'outside', 'outside'], textfont=dict(size=25))

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', '#A0D568', '#4fc1e8', '#ac92eb']
bar_line_colors = ['grey', '#ffce54', '#4fc1e8', '#ac92eb']
bar_line_widths = [4, 0, 0, 0]
text_colors = ['grey', 'white', '#4fc1e8', '#ac92eb']

# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))


# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=3.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)

fig.show()

In [ ]:
# @title • Ad CVR for Partner A
# Filter ads_df for Partner A
partner_a_ads_df = ads_df[ads_df['Partner'] == 'Partner A'].copy()

# Calculate Ad CVR for each media type for Partner A
partner_a_media_ad_cvr = partner_a_ads_df.groupby('Media').apply(lambda x: x['Ad Sold Units'].sum() / x['Impressions'].sum(), include_groups=False).reset_index(name='Ad CVR')

# Calculate overall Ad CVR for all media for Partner A
partner_a_overall_ad_cvr = partner_a_ads_df['Ad Sold Units'].sum() / partner_a_ads_df['Impressions'].sum()

# Create a DataFrame for plotting including overall Ad CVR for Partner A
partner_a_ad_cvr_data = pd.DataFrame({
    'Media': ['Overall'] + partner_a_media_ad_cvr['Media'].tolist(),
    'Ad CVR': [partner_a_overall_ad_cvr] + partner_a_media_ad_cvr['Ad CVR'].tolist()
})

# Create the bar chart
fig = px.bar(partner_a_ad_cvr_data,
             x='Media',
             y='Ad CVR',
             title='Ad CVR by Media Type for Partner A',
             text=partner_a_ad_cvr_data['Ad CVR'].apply(lambda x: f'<b>{x:.2%}</b>')) # Add text to show values on bars with 2 decimal places


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Media Type', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Ad CVR by Media Type for Partner A</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=900, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition=['inside', 'inside', 'inside', 'outside'], textfont=dict(size=25)) # Changed text color to white

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', '#A0D568', '#4fc1e8', '#ac92eb']
bar_line_colors = ['grey', '#ffce54', '#4fc1e8', '#ac92eb']
bar_line_widths = [4, 0, 0, 0]
text_colors = ['grey', 'white', 'white', '#ac92eb'] # Changed text color for the last bar

# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))


# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=3.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

In [ ]:
# @title • Ad CVR for Partner B
# Filter ads_df for Partner B
partner_b_ads_df = ads_df[ads_df['Partner'] == 'Partner B'].copy()

# Calculate Ad CVR for each media type for Partner B
partner_b_media_ad_cvr = partner_b_ads_df.groupby('Media').apply(lambda x: x['Ad Sold Units'].sum() / x['Impressions'].sum(), include_groups=False).reset_index(name='Ad CVR')

# Calculate overall Ad CVR for all media for Partner B
partner_b_overall_ad_cvr = partner_b_ads_df['Ad Sold Units'].sum() / partner_b_ads_df['Impressions'].sum()

# Create a DataFrame for plotting including overall Ad CVR for Partner B
partner_b_ad_cvr_data = pd.DataFrame({
    'Media': ['Overall'] + partner_b_media_ad_cvr['Media'].tolist(),
    'Ad CVR': [partner_b_overall_ad_cvr] + partner_b_media_ad_cvr['Ad CVR'].tolist()
})

# Create the bar chart
fig = px.bar(partner_b_ad_cvr_data,
             x='Media',
             y='Ad CVR',
             title='Ad CVR by Media Type for Partner B',
             text=partner_b_ad_cvr_data.apply(lambda row: f'<b>{row["Ad CVR"]:.3%}</b>' if row['Media'] == 'Meta' else f'<b>{row["Ad CVR"]:.2%}</b>', axis=1)) # Apply conditional formatting


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Media Type', font=dict(size=18, weight='bold')), # Make x-axis title bigger and bold
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Ad CVR by Media Type for Partner B</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=500, width=900, # Adjusted height and width
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=18, weight='bold')) # Make x-axis tick labels bigger and bold
)

# Update text position on bars
fig.update_traces(textposition=['inside', 'inside', 'outside', 'inside'], textfont=dict(size=25)) # Changed text color to white

# Define colors, border colors, border widths, and text colors for each bar
bar_colors = ['white', '#A0D568', '#4fc1e8', '#ac92eb']
bar_line_colors = ['grey', '#ffce54', '#4fc1e8', '#ac92eb']
bar_line_widths = [4, 0, 0, 0]
text_colors = ['grey', 'white', '#4fc1e8', 'white'] # Changed text color for the last bar

# Update bar colors, borders, and text colors using lists
fig.update_traces(marker_color=bar_colors,
                  marker_line_color=bar_line_colors,
                  marker_line_width=bar_line_widths,
                  textfont=dict(color=text_colors))


# Add a line at the bottom of the bars
fig.add_shape(
    type="line",
    x0=-0.5,  # Start at the left edge of the first bar
    y0=0,     # Start at the bottom of the y-axis
    x1=3.5,   # End at the right edge of the last bar
    y1=0,     # End at the bottom of the y-axis
    line=dict(
        color="grey",
        width=2,
    )
)


fig.show()

## 8.2 Media Composition

In [ ]:
# @title • Impression Share
# Calculate total impressions for each media type
media_impressions = ads_df.groupby('Media')['Impressions'].sum().reset_index()

# Calculate the total impressions across all media
total_impressions = media_impressions['Impressions'].sum()

# Calculate the percentage of impressions for each media type
media_impressions['Percentage'] = media_impressions['Impressions'] / total_impressions

# Create a dummy column for stacking
media_impressions['All Media'] = 'All Media'

# Create the stacked horizontal bar chart
fig = px.bar(media_impressions,
             x='Percentage',
             y='All Media',  # Use a single category for stacking
             color='Media',
             orientation='h',
             title='Percentage of Impressions by Media Type (Stacked)',
             text=media_impressions['Percentage'].apply(lambda x: f'<b>{x:.1%}</b>'), # Add text to show percentages
             color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Percentage', font=dict(size=18, weight='bold')),
    yaxis_title='', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Percentage of Impressions by Media Type</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=300, width=1000, # Adjusted height and width for a single bar
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=14)), # Adjust x-axis tick font size
    showlegend=True, # Show legend for media types
    legend_title_text='Media Type' # Set legend title
)

# Update text position and add white border
fig.update_traces(textposition='inside', textfont=dict(size=25, color='white'), insidetextanchor='middle',
                  marker=dict(line=dict(color='white', width=2)))


fig.show()

In [ ]:
# @title • Impression Share for Partner A
# Filter ads_df for Partner A
partner_a_ads_df = ads_df[ads_df['Partner'] == 'Partner A'].copy()

# Calculate total impressions for each media type for Partner A
partner_a_media_impressions = partner_a_ads_df.groupby('Media')['Impressions'].sum().reset_index()

# Calculate the total impressions across all media for Partner A
partner_a_total_impressions = partner_a_media_impressions['Impressions'].sum()

# Calculate the percentage of impressions for each media type for Partner A
partner_a_media_impressions['Percentage'] = partner_a_media_impressions['Impressions'] / partner_a_total_impressions

# Create a dummy column for stacking
partner_a_media_impressions['All Media'] = 'All Media'

# Create the stacked horizontal bar chart for Partner A
fig = px.bar(partner_a_media_impressions,
             x='Percentage',
             y='All Media',  # Use a single category for stacking
             color='Media',
             orientation='h',
             title='Percentage of Impressions by Media Type for Partner A (Stacked)',
             text=partner_a_media_impressions['Percentage'].apply(lambda x: f'<b>{x:.1%}</b>'), # Add text to show percentages
             color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Percentage', font=dict(size=18, weight='bold')),
    yaxis_title='', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Percentage of Impressions by Media Type for Partner A</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=300, width=1000, # Adjusted height and width for a single bar
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=14)), # Adjust x-axis tick font size
    showlegend=True, # Show legend for media types
    legend_title_text='Media Type' # Set legend title
)

# Update text position and add white border
fig.update_traces(textposition='inside', textfont=dict(size=25, color='white'), insidetextanchor='middle',
                  marker=dict(line=dict(color='white', width=2)))


fig.show()

In [ ]:
# @title •Impressions Share for Partner B
# Filter ads_df for Partner B
partner_b_ads_df = ads_df[ads_df['Partner'] == 'Partner B'].copy()

# Calculate total impressions for each media type for Partner B
partner_b_media_impressions = partner_b_ads_df.groupby('Media')['Impressions'].sum().reset_index()

# Calculate the total impressions across all media for Partner B
partner_b_total_impressions = partner_b_media_impressions['Impressions'].sum()

# Calculate the percentage of impressions for each media type for Partner B
partner_b_media_impressions['Percentage'] = partner_b_media_impressions['Impressions'] / partner_b_total_impressions

# Create a dummy column for stacking
partner_b_media_impressions['All Media'] = 'All Media'

# Create the stacked horizontal bar chart for Partner B
fig = px.bar(partner_b_media_impressions,
             x='Percentage',
             y='All Media',  # Use a single category for stacking
             color='Media',
             orientation='h',
             title='Percentage of Impressions by Media Type for Partner B (Stacked)',
             text=partner_b_media_impressions['Percentage'].apply(lambda x: f'<b>{x:.1%}</b>'), # Add text to show percentages
             color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb'})


# Update layout
fig.update_layout(
    xaxis_title=dict(text='Percentage', font=dict(size=18, weight='bold')),
    yaxis_title='', # Remove y-axis title
    yaxis=dict(showticklabels=False), # Remove y-axis tick labels
    title=dict(
        text='<b>Percentage of Impressions by Media Type for Partner B</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    height=300, width=1000, # Adjusted height and width for a single bar
    plot_bgcolor='rgba(0,0,0,0)', # Remove background
    xaxis=dict(tickfont=dict(size=14)), # Adjust x-axis tick font size
    showlegend=True, # Show legend for media types
    legend_title_text='Media Type' # Set legend title
)

# Update text position and add white border
fig.update_traces(textposition='inside', textfont=dict(size=25, color='white'), insidetextanchor='middle',
                  marker=dict(line=dict(color='white', width=2)))


fig.show()

## 8.3 Pie Charts for Sold Units

In [ ]:
# @title • Sold Units by Media

import plotly.express as px
import pandas as pd

# Calculate the total Ad Sold Units for each media type
media_ad_sold_units = ads_df.groupby('Media')['Ad Sold Units'].sum().reset_index()

# Calculate the total 'Total Sold Units'
total_sold_units = merged_df['Total Sold Units'].sum()

# Calculate the total 'Ad Sold Units'
total_ad_sold_units = media_ad_sold_units['Ad Sold Units'].sum()

# Calculate the 'Others' value
others_sold_units = total_sold_units - total_ad_sold_units

# Create a DataFrame for plotting including the 'Others' category
ad_sold_units_data = pd.DataFrame({
    'Media': media_ad_sold_units['Media'].tolist() + ['Others*'],
    'Ad Sold Units': media_ad_sold_units['Ad Sold Units'].tolist() + [others_sold_units]
})

# Create the donut chart
fig = px.pie(ad_sold_units_data,
             values='Ad Sold Units',
             names='Media',
             title='Distribution of Total Sold Units by Source (Ads and Others)',
             hole=0.5,  # This creates the donut hole
             color='Media',
             color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb', 'Others*': '#f2f2f2'}) # Changed color for 'Others' to very light grey


# Update layout for better appearance
fig.update_layout(
    title=dict(
        text='<b>Distribution of Total Sold Units by Source (Ads and Others)</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    legend_title_text='Source', # Set legend title
    height=700, width=1000 # Set chart size
)

# Update traces for text information, white border, and include name, and format value with comma
fig.update_traces(textinfo='percent+value+label', textposition='inside',
                  marker=dict(line=dict(color='white', width=2)),
                  hovertemplate='<b>%{label}</b><br>Sold Units: %{value:,}<br>Percentage: %{percent}',
                  texttemplate='<b>%{label}<br>%{percent}<br>%{value:,}</b>',
                  pull=[0, 0, 0, 0.1]) # Pull out the 'Others' slice (assuming it's the last one)

# Define text colors for each slice
text_colors = ['white'] * len(media_ad_sold_units) + ['grey'] # Set text color for 'Others' to grey

# Update text colors for each slice
fig.update_traces(textfont=dict(color=text_colors, size=14))

# Update marker line color for Others
fig.update_traces(marker_line_color=['white'] * len(media_ad_sold_units) + ['lightgrey'], marker_line_width=2)


fig.show()

In [ ]:
# @title • Sold Units by Media for Partner A
import plotly.express as px
import pandas as pd

# Filter data for Partner A
partner_a_ads_df = ads_df[ads_df['Partner'] == 'Partner A'].copy()
partner_a_merged_df = merged_df[merged_df['Partner'] == 'Partner A'].copy()

# Calculate the total Ad Sold Units for each media type for Partner A
partner_a_media_ad_sold_units = partner_a_ads_df.groupby('Media')['Ad Sold Units'].sum().reset_index()

# Calculate the total 'Total Sold Units' for Partner A
partner_a_total_sold_units = partner_a_merged_df['Total Sold Units'].sum()

# Calculate the total 'Ad Sold Units' for Partner A
partner_a_total_ad_sold_units = partner_a_media_ad_sold_units['Ad Sold Units'].sum()

# Calculate the 'Others' value for Partner A
partner_a_others_sold_units = partner_a_total_sold_units - partner_a_total_ad_sold_units

# Create a DataFrame for plotting including the 'Others' category for Partner A
partner_a_ad_sold_units_data = pd.DataFrame({
    'Media': partner_a_media_ad_sold_units['Media'].tolist() + ['Others*'],
    'Ad Sold Units': partner_a_media_ad_sold_units['Ad Sold Units'].tolist() + [partner_a_others_sold_units]
})

# Create the donut chart for Partner A
fig = px.pie(partner_a_ad_sold_units_data,
             values='Ad Sold Units',
             names='Media',
             title='Distribution of Total Sold Units by Source for Partner A (Ads and Others)',
             hole=0.5,  # This creates the donut hole
             color='Media',
             color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb', 'Others*': '#f2f2f2'})


# Update layout for better appearance
fig.update_layout(
    title=dict(
        text='<b>Distribution of Total Sold Units by Source for Partner A (Ads and Others)</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    legend_title_text='Source', # Set legend title
    height=700, width=1000 # Set chart size
)

# Update traces for text information, white border, and include name, and format value with comma
fig.update_traces(textinfo='percent+value+label', textposition='inside',
                  marker=dict(line=dict(color='white', width=2)),
                  hovertemplate='<b>%{label}</b><br>Sold Units: %{value:,}<br>Percentage: %{percent}',
                  texttemplate='<b>%{label}<br>%{percent}<br>%{value:,}</b>',
                  pull=[0, 0, 0, 0.1]) # Pull out the 'Others' slice (assuming it's the last one)

# Define text colors for each slice
text_colors = ['white'] * len(partner_a_media_ad_sold_units) + ['grey'] # Set text color for 'Others' to grey

# Update text colors for each slice
fig.update_traces(textfont=dict(color=text_colors, size=14))

# Update marker line color for Others
fig.update_traces(marker_line_color=['white'] * len(partner_a_media_ad_sold_units) + ['lightgrey'], marker_line_width=2)


fig.show()

In [ ]:
# @title • Sold Units by Media for Partner B
import plotly.express as px
import pandas as pd

# Filter data for Partner B
partner_b_ads_df = ads_df[ads_df['Partner'] == 'Partner B'].copy()
partner_b_merged_df = merged_df[merged_df['Partner'] == 'Partner B'].copy()

# Calculate the total Ad Sold Units for each media type for Partner B
partner_b_media_ad_sold_units = partner_b_ads_df.groupby('Media')['Ad Sold Units'].sum().reset_index()

# Calculate the total 'Total Sold Units' for Partner B
partner_b_total_sold_units = partner_b_merged_df['Total Sold Units'].sum()

# Calculate the total 'Ad Sold Units' for Partner B
partner_b_total_ad_sold_units = partner_b_media_ad_sold_units['Ad Sold Units'].sum()

# Calculate the 'Others' value for Partner B
partner_b_others_sold_units = partner_b_total_sold_units - partner_b_total_ad_sold_units

# Create a DataFrame for plotting including the 'Others' category for Partner B
partner_b_ad_sold_units_data = pd.DataFrame({
    'Media': partner_b_media_ad_sold_units['Media'].tolist() + ['Others*'],
    'Ad Sold Units': partner_b_media_ad_sold_units['Ad Sold Units'].tolist() + [partner_b_others_sold_units]
})

# Create the donut chart for Partner B
fig = px.pie(partner_b_ad_sold_units_data,
             values='Ad Sold Units',
             names='Media',
             title='Distribution of Total Sold Units by Source for Partner B (Ads and Others)',
             hole=0.5,  # This creates the donut hole
             color='Media',
             color_discrete_map={'Google': '#A0D568', 'Meta': '#4fc1e8', 'TikTok': '#ac92eb', 'Others*': '#f2f2f2'})


# Update layout for better appearance
fig.update_layout(
    title=dict(
        text='<b>Distribution of Total Sold Units by Source for Partner B (Ads and Others)</b>',
        xanchor='center',
        x=0.5,
        font=dict(size=20)
    ),
    legend_title_text='Source', # Set legend title
    height=700, width=1000 # Set chart size
)

# Update traces for text information, white border, and include name, and format value with comma
fig.update_traces(textinfo='percent+value+label', textposition='inside',
                  marker=dict(line=dict(color='white', width=2)),
                  hovertemplate='<b>%{label}</b><br>Sold Units: %{value:,}<br>Percentage: %{percent}',
                  texttemplate='<b>%{label}<br>%{percent}<br>%{value:,}</b>',
                  pull=[0, 0, 0, 0.1]) # Pull out the 'Others' slice (assuming it's the last one)

# Define text colors for each slice
text_colors = ['white'] * len(partner_b_media_ad_sold_units) + ['grey'] # Set text color for 'Others' to grey

# Update text colors for each slice
fig.update_traces(textfont=dict(color=text_colors, size=14))

# Update marker line color for Others
fig.update_traces(marker_line_color=['white'] * len(partner_b_media_ad_sold_units) + ['lightgrey'], marker_line_width=2)


fig.show()

In [ ]:
#--- END OF THE NOTEBOOK ---